In [1]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import py1_data_clean as pydc

dictionary = pydc.dictionary
matrix = pydc.matrix
matrix_train = pydc.matrix_train
matrix_test = pydc.matrix_test

rules = pd.read_pickle('rules_minsup0.003_maxlen4.pkl')
rules = rules[rules['confidence'] >= 0.5]
rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
381382,"(配偶者の幸福感（最近１年間）_0, 新しい技術の普及で生じることの抵抗感（1）_3, 市郡...",(本人の幸福感（最近１年間）_5),0.009948,0.213643,0.005211,0.523810,2.451800,1.0,0.003086,1.651350,0.598086,0.023861,0.394435,0.274100
383530,"(配偶者の幸福感（最近１年間）_0, 新しい技術の普及で生じることの抵抗感（1）_3, 就業...",(本人の幸福感（最近１年間）_5),0.025580,0.213643,0.012790,0.500000,2.340355,1.0,0.007325,1.572714,0.587749,0.056485,0.364157,0.279933
424599,"(仕事で対面で話をすること（3）_3, 配偶者の幸福感（最近１年間）_0, 新しい技術の普及...",(本人の幸福感（最近１年間）_5),0.012316,0.213643,0.006158,0.500000,2.340355,1.0,0.003527,1.572714,0.579856,0.028017,0.364157,0.264412
447459,"(配偶者の幸福感（最近１年間）_0, 新しい技術の普及で生じることの抵抗感（1）_3, 飲酒...",(本人の幸福感（最近１年間）_5),0.010422,0.213643,0.005211,0.500000,2.340355,1.0,0.002984,1.572714,0.578746,0.023810,0.364157,0.262195
465689,"(配偶者の幸福感（最近１年間）_0, 新しい技術の普及で生じることの抵抗感（1）_3, タバ...",(本人の幸福感（最近１年間）_5),0.010895,0.213643,0.005685,0.521739,2.442109,1.0,0.003357,1.644201,0.597023,0.025974,0.391802,0.274173


相関ルールマイニングを踏まえて推薦を行う

In [ ]:
# 準備① : rulesをもとに, 各ルールの前提部・結論部(幸福度)・確信度から成るpd.Dataframe「rules_df」を作成

# 処理結果を格納するためのリスト
rules_list = []

for antecedents, consequents, confidence in rules[['antecedents', 'consequents', 'confidence']].values:
    
    # 各ルール（DataFrameの将来の1行）を辞書として作成
    rule_dict = {}
    
    # 確信度を辞書に追加
    rule_dict['確信度'] = confidence
    
    # 前提部（antecedents）と結論部（consequents）を結合
    all_items = antecedents.union(consequents)
    
    for item in all_items:
        # アイテムが 'カラム名_値' の形式であるかチェック
        if isinstance(item, str) and '_' in item:
            # カラム名と値に分割
            column_name, value_str = item.split('_')
            value = int(float(value_str))
                
            # 辞書に {カラム名: 値} をセット
            rule_dict[column_name] = value
    
    # 完成した辞書をリストに追加
    rules_list.append(rule_dict)

# 辞書のリストからDataFrameを作成
columns_order = list(dictionary.keys()) + ['確信度']
rules_df = pd.DataFrame(rules_list, columns=columns_order)

rules_df.head()

,配偶者の有無,対象者性別,対象者生年月日（生年）,同居人数,一年前の居住,世帯変動・子ども,世帯員が単身赴任から戻る,世帯員が単身赴任する,世帯変動・転出,世帯変動・死亡,...,飲酒習慣,タバコの喫煙,通勤通学以外で運動する日数,介護を必要とする家族,平日睡眠時間（平均時間）,休日睡眠時間（平均時間）,配偶者の幸福感（最近１年間）,地域ブロック,市郡規模,確信度
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,1.0,0.523810
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.500000
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.500000
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.500000
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.521739


In [4]:
#準備② : 「profiles_df」と, それに対応するworried_columnを作成

profiles_df = matrix_test.copy()

#profileに欠損値や非該当値が含まれている場合, Noneに置換する
for index, row in profiles_df.iterrows():
    for col_name, value in row.items():
        
        if value in list(dictionary[col_name]):
            continue
        else:
            profiles_df.loc[index, col_name] = None

worried_column = '配偶者の有無' #今回の設定③

In [5]:
#準備③ : happiness_estimationを実行するための決まった処理を行う

#rules_dfの, 幸福度と確信度以外の列から成る「rules_df_rules」を作成する
rules_df_rules = rules_df.drop(columns=['本人の幸福感（最近１年間）', '確信度'])

#各ルールの確信度が格納されたリスト「confidence_list」を作成する
rules_df_confidence = rules_df['確信度']
confidence_list = rules_df_confidence.tolist()
confidence_list = [min(0.7, conf) for conf in confidence_list]

#rules_dfにおける「幸福度」の値の頻度を格納
frequency_dict = rules_df['本人の幸福感（最近１年間）'].value_counts().to_dict()
frequency_dict_max = max ( value_list for value_list in frequency_dict.values() )
frequency_dict_min = min ( value_list for value_list in frequency_dict.values() )

In [6]:
def happiness_estimation(profile_df):

  # プロフィールの値 (1D配列)
  profile_df_main = profile_df.drop(columns='本人の幸福感（最近１年間）')
  profile_values = profile_df_main.iloc[0].values # (n_features,)
  
  # 比較対象の列名
  feature_names = profile_df_main.columns.tolist()
  
  # ルールの値 (2D配列)
  rules_values = rules_df_rules[feature_names].values # (n_rules, n_features)
  
  
  # ①各ルールのプロフィールとのマッチ度が格納されたリスト「matching_list」を作成する

  # (a) 各属性の範囲 (a_under) を事前に一括計算 (1D配列)
  ranges = np.array([
      max(dictionary[col]) - min(dictionary[col]) for col in feature_names
  ])
  
  # (b) ゼロ除算対策
  # rangesが0の場所は分母を1にし(safe_ranges)、
  # 該当箇所を後で1.0で上書きするためにマスク(zero_range_mask)を作成
  safe_ranges = np.where(ranges == 0, 1.0, ranges)
  zero_range_mask = (ranges == 0) # (n_features,)
  
  # (c) カテゴリカル属性のマスク (1D配列)
  categorical_features_list = [
      '両親の生死', '技術・技能の習得', '副業の有無', '仕事の内容', '経営組織', '職位', 
      '働き方', '現在の仕事の継続', '仕事を変えたい理由', '１年前の就業', 
      '通勤通学以外で運動する日数', '介護を必要とする家族', '地域ブロック', '市群規模'
  ]
  categorical_mask = np.array([col in categorical_features_list for col in feature_names])

  # (d) NaNのチェック (2Dマスク)
  # ルールとプロフィールの両方がNaNでないことを確認
  notna_mask = ~np.isnan(rules_values) & ~np.isnan(profile_values) # (n_rules, n_features)

  
  # (e) マッチ度を一括計算 
  
  # (e-1) 数値属性としての計算: 1 - abs(rule - profile) / range
  numerical_matching = 1.0 - (np.abs(rules_values - profile_values) / safe_ranges)
  
  # (e-2) カテゴリカル属性としての計算 
  # 一致: 1.0
  # 不一致: 1.0 - (1.0 / safe_ranges)
  categorical_matching = np.where(
      rules_values == profile_values, 
      1.0,                            
      1.0 - (1.0 / safe_ranges)       
  )

  # (e-3) カテゴリカル属性の箇所だけ計算結果を差し替え
  matching_scores = np.where(
      categorical_mask,     
      categorical_matching, 
      numerical_matching    
  )

  # (e-4) a_under=0 (zero_range_mask) の箇所を 1.0 で上書き
  matching_scores = np.where(
      zero_range_mask,       
      1.0,                  
      matching_scores       
  )
  
  # (e-5) NaNだった箇所を 0 に (合計に寄与させない)
  final_matching_scores = np.where(notna_mask, matching_scores, 0.0)

  # (f) 各ルール（行ごと）の合計を計算 (axis=1)
  matching_list = np.sum(final_matching_scores, axis=1)


  # ②各ルールの「confidence_list」と「matching_list」から計算した「importance_list」が格納されたリスト
  importance_list = [confidence * matching for confidence, matching in zip(confidence_list, matching_list)]

  # 推定幸福度を計算
  estimated_happiness_apper = 0 #推定幸福度を計算する加重平均の式における, 分子
  estimated_happiness_under = sum(importance_list) #推定幸福度を計算する加重平均の式における, 分母

  for i in range(len(importance_list)):
    estimated_happiness_apper += importance_list[i] * rules_df.iloc[i]['本人の幸福感（最近１年間）']

  estimated_happiness = estimated_happiness_apper / estimated_happiness_under

  return estimated_happiness

In [7]:
# 精度計算（推定幸福度を常に, 全幸福度の平均とした場合）

squared_errors = []
for index in range(len(profiles_df)):
    estimated_happiness = rules_df['本人の幸福感（最近１年間）'].mean()
    correct_happiness = profiles_df.iloc[index, :].to_frame().T['本人の幸福感（最近１年間）'].iloc[0]
    squared_error = (estimated_happiness - correct_happiness) ** 2
    squared_errors.append(squared_error)
rmse = np.sqrt( np.nanmean(squared_errors) )
print( 'RMSE:', rmse )

RMSE: 2.2422792047486926


In [8]:
# 精度計算（RMSE）

squared_errors = []
for index in range(len(profiles_df)):
    estimated_happiness = happiness_estimation(profiles_df.iloc[index, :].to_frame().T)
    correct_happiness = profiles_df.iloc[index, :].to_frame().T['本人の幸福感（最近１年間）'].iloc[0]
    squared_error = (estimated_happiness - correct_happiness) ** 2
    squared_errors.append(squared_error)
rmse = np.sqrt( np.nanmean(squared_errors) )
print( 'RMSE:', rmse )

RMSE: 2.19572008897113


In [11]:
# # 検証用

# print( '平均幸福度は', profiles_df['本人の幸福感（最近１年間）'].mean() )

# for changed_index in np.arange(16):
#     print( '現在の推定幸福度は', happiness_estimation(profiles_df.iloc[changed_index, :].to_frame().T) )
#     print( '現在の正解幸福度は', profiles_df.iloc[changed_index, :].to_frame().T['本人の幸福感（最近１年間）'].iloc[0] )

以下実行せず

In [ ]:

for index, profile_df in profiles_df.iterrows():
  profile_df = profile_df.to_frame().T
  estimated_happiness = happiness_estimation(profile_df)
  print(estimated_happiness) #精度の計算には邪魔なので一時的にコメントアウト

  #精度を計算
  accuracy.append( estimated_happiness - profile_df['本人の幸福感（最近１年間）'].iloc[0] )

  recommendations = []

  #様々なprofile_dfを作成して, その度にhappiness_estimation関数を実行する
  for worried_column_value in dictionary[worried_column]:
    #まずは, 悩んでいる属性の属性値を変えたときのprofileをDataFrame化する
    new_profile_df = profile_df.copy()
    new_profile_df[worried_column] = worried_column_value
  
    if new_profile_df[worried_column].iloc[0] != profile_df[worried_column].iloc[0]:
      new_estimated_happiness = happiness_estimation(new_profile_df)
      print(new_estimated_happiness) #精度の計算には邪魔なので一時的にコメントアウト

      if new_estimated_happiness > estimated_happiness:
        recommendations.append(worried_column_value) #あとで, 推定幸福度が大きいものから順にrecommendationsに格納するようにする

  if recommendations == []:
    print('推薦内容はありません')
  else:
    print(recommendations)

#精度の計算
mean_squared_error = sum(x**2 for x in accuracy) / len(accuracy)
print('Mean Squared Error:', mean_squared_error)

6.130324254757696
6.136571931690092
[1]
6.130659082986812
6.137631173233993
[1]
6.1134020434693905
6.104143477417357
推薦内容はありません
6.0696423794991485
6.053091554688825
推薦内容はありません
6.138766339191197
6.130205913947218
推薦内容はありません
6.121140813042228
6.1128418881517925
推薦内容はありません
6.134068210903929
6.1279372908296885
推薦内容はありません
6.147027820907052
6.140506515166645
推薦内容はありません
6.124377037675915
6.116465819374055
推薦内容はありません
6.142894345374708
6.136812862506486
推薦内容はありません
6.119602162115379
6.1122092426517565
推薦内容はありません
6.091102039937346
6.080478058708889
推薦内容はありません
6.141196833057147
6.1358013856981035
推薦内容はありません
6.119414323846997
6.126085993106332
[1]
6.099526548556365
6.107812663499592
[1]
6.117776117574677
6.109672421222824
推薦内容はありません
6.115602883743836
6.126908763878944
[1]
6.124271182205654
6.115698728157486
推薦内容はありません
6.087517126871693
6.072715249255991
推薦内容はありません
6.157958813814232
6.15000588340394
推薦内容はありません
6.127137370214473
6.13484597616176
[1]
6.116051378291546
6.108379521690532
推薦内容はありません
6.13